<a href="https://colab.research.google.com/github/MobiNomi/Complete-ML-Pipeline/blob/main/Sentinel_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install kaggle


In [3]:
import os

os.environ['KAGGLE_USERNAME'] = "MubashirRana"
os.environ['KAGGLE_KEY'] = "KGAT_351a8983a8482370529371da2bec0198"

In [4]:
!kaggle datasets download -d apollo2506/eurosat-dataset

Dataset URL: https://www.kaggle.com/datasets/apollo2506/eurosat-dataset
License(s): CC0-1.0
100% 2.04G/2.04G [00:10<00:00, 216MB/s]



In [5]:
!ls

eurosat-dataset.zip  sample_data


In [6]:
import zipfile

with zipfile.ZipFile("eurosat-dataset.zip", 'r') as zip_ref:
    zip_ref.extractall("/content/eurosat")

In [7]:
import os

os.listdir("/content/eurosat")

['EuroSATallBands', 'EuroSAT']

In [8]:
import os

os.listdir('/content/eurosat/EuroSAT')

['SeaLake',
 'Residential',
 'label_map.json',
 'Pasture',
 'Highway',
 'test.csv',
 'AnnualCrop',
 'River',
 'validation.csv',
 'PermanentCrop',
 'Forest',
 'Industrial',
 'HerbaceousVegetation',
 'train.csv']

In [9]:
classes = os.listdir('/content/eurosat/EuroSAT')

print("Number of classes:", len(classes))
print(classes)

Number of classes: 14
['SeaLake', 'Residential', 'label_map.json', 'Pasture', 'Highway', 'test.csv', 'AnnualCrop', 'River', 'validation.csv', 'PermanentCrop', 'Forest', 'Industrial', 'HerbaceousVegetation', 'train.csv']


In [10]:
import os

dataset_path = "/content/eurosat/EuroSAT"

for item in os.listdir(dataset_path):
    full_path = os.path.join(dataset_path, item)

    if os.path.isdir(full_path):
        print(item)

SeaLake
Residential
Pasture
Highway
AnnualCrop
River
PermanentCrop
Forest
Industrial
HerbaceousVegetation


In [11]:
import os

for item in os.listdir('/content/eurosat/EuroSAT'):
    print(item)

SeaLake
Residential
label_map.json
Pasture
Highway
test.csv
AnnualCrop
River
validation.csv
PermanentCrop
Forest
Industrial
HerbaceousVegetation
train.csv


In [12]:
import pandas as pd

train_df = pd.read_csv('/content/eurosat/EuroSAT/train.csv')

train_df.head()

,Unnamed: 0,Filename,Label,ClassName
0,16257,AnnualCrop/AnnualCrop_142.jpg,0,AnnualCrop
1,3297,HerbaceousVegetation/HerbaceousVegetation_2835...,2,HerbaceousVegetation
2,17881,PermanentCrop/PermanentCrop_1073.jpg,6,PermanentCrop
3,2223,Industrial/Industrial_453.jpg,4,Industrial
4,4887,HerbaceousVegetation/HerbaceousVegetation_1810...,2,HerbaceousVegetation


In [14]:
import pandas as pd

train_df = pd.read_csv('/content/eurosat/EuroSAT/train.csv')

print(train_df.shape)

(18900, 4)


In [15]:
val_df = pd.read_csv('/content/eurosat/EuroSAT/validation.csv')
test_df = pd.read_csv('/content/eurosat/EuroSAT/test.csv')

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

Train: (18900, 4)
Validation: (5400, 4)
Test: (2700, 4)


In [17]:
# =====================================
# 1. IMPORTS
# =====================================

import os
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from torchvision import transforms
from torchvision import models

# =====================================
# 2. DEVICE
# =====================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", device)

# =====================================
# 3. DATA PATHS
# =====================================

ROOT_DIR = "/content/eurosat/EuroSAT"

TRAIN_CSV = os.path.join(ROOT_DIR, "train.csv")
VAL_CSV = os.path.join(ROOT_DIR, "validation.csv")
TEST_CSV = os.path.join(ROOT_DIR, "test.csv")

# =====================================
# 4. IMAGE TRANSFORMS
# =====================================

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# =====================================
# 5. CUSTOM DATASET
# =====================================

class EuroSATDataset(Dataset):

    def __init__(self, csv_file, root_dir, transform=None):

        self.data = pd.read_csv(csv_file)

        self.root_dir = root_dir

        self.transform = transform

    def __len__(self):

        return len(self.data)

    def __getitem__(self, idx):

        img_relative_path = self.data.iloc[idx]["Filename"]

        label = self.data.iloc[idx]["Label"]

        img_path = os.path.join(
            self.root_dir,
            img_relative_path
        )

        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, label

# =====================================
# 6. DATASETS
# =====================================

train_dataset = EuroSATDataset(
    TRAIN_CSV,
    ROOT_DIR,
    transform
)

val_dataset = EuroSATDataset(
    VAL_CSV,
    ROOT_DIR,
    transform
)

test_dataset = EuroSATDataset(
    TEST_CSV,
    ROOT_DIR,
    transform
)

# =====================================
# 7. DATALOADERS
# =====================================

BATCH_SIZE = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

# =====================================
# 8. LOAD PRETRAINED RESNET18
# =====================================

model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

# =====================================
# 9. FREEZE ALL LAYERS
# =====================================

for param in model.parameters():
    param.requires_grad = False

# =====================================
# 10. REPLACE FC LAYER
# =====================================

num_features = model.fc.in_features

model.fc = nn.Linear(
    num_features,
    10
)

# =====================================
# 11. MOVE MODEL TO GPU
# =====================================

model = model.to(device)

# =====================================
# 12. LOSS FUNCTION
# =====================================

criterion = nn.CrossEntropyLoss()

# =====================================
# 13. OPTIMIZER
# =====================================

optimizer = torch.optim.Adam(
    model.fc.parameters(),
    lr=0.001
)

# =====================================
# 14. TRAINING LOOP
# =====================================

EPOCHS = 7

best_val_acc = 0

for epoch in range(EPOCHS):

    model.train()

    running_loss = 0
    correct = 0
    total = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)

        correct += (predicted == labels).sum().item()

    train_acc = 100 * correct / total

    # ======================
    # VALIDATION
    # ======================

    model.eval()

    val_correct = 0
    val_total = 0

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            _, predicted = torch.max(outputs, 1)

            val_total += labels.size(0)

            val_correct += (predicted == labels).sum().item()

    val_acc = 100 * val_correct / val_total

    print(
        f"Epoch [{epoch+1}/{EPOCHS}] "
        f"Loss: {running_loss:.4f} "
        f"Train Acc: {train_acc:.2f}% "
        f"Val Acc: {val_acc:.2f}%"
    )

    if val_acc > best_val_acc:

        best_val_acc = val_acc

        torch.save(
            model.state_dict(),
            "best_resnet18_eurosat.pth"
        )

# =====================================
# 15. TESTING
# =====================================

model.load_state_dict(
    torch.load("best_resnet18_eurosat.pth")
)

model.eval()

test_correct = 0
test_total = 0

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        _, predicted = torch.max(outputs, 1)

        test_total += labels.size(0)

        test_correct += (predicted == labels).sum().item()

test_accuracy = 100 * test_correct / test_total

print(f"\nTest Accuracy: {test_accuracy:.2f}%")

Using device: cuda
Epoch [1/7] Loss: 355.5935 Train Acc: 82.26% Val Acc: 90.07%
Epoch [2/7] Loss: 192.2734 Train Acc: 89.43% Val Acc: 91.56%
Epoch [3/7] Loss: 172.2810 Train Acc: 90.25% Val Acc: 92.57%
Epoch [4/7] Loss: 156.6245 Train Acc: 90.77% Val Acc: 92.11%
Epoch [5/7] Loss: 152.1323 Train Acc: 91.16% Val Acc: 92.61%
Epoch [6/7] Loss: 141.9091 Train Acc: 91.48% Val Acc: 92.78%
Epoch [7/7] Loss: 137.4198 Train Acc: 92.02% Val Acc: 92.26%

Test Accuracy: 93.89%
